# Probabilistic transformer head study

This notebook compares different output heads for the Probabilistic Transformer:
- Gaussian
- Johnson SU
- Mixture of Gaussians (1, 2, 3, 5 components)
- Mixture of Johnson SU (1, 2, 3, 5 components)
- Quantile Regression (9, 19, ..., 99 quantiles)

Each configuration is run 10 times to account for statistical variability.

## Metrics
- **Point**: MAE, RMSE, MAPE
- **Probabilistic**: PICP, MPIW, PINAW, Interval Score, CRPS
- **Quantile**: Pinball losses (10th, 50th, 90th percentiles)

In [1]:
import os
import sys
import json
import hashlib
import time
import gc
from pathlib import Path
from typing import Dict, Any, List

import numpy as np
import pandas as pd
import tensorflow as tf

# Project root setup
current_dir = Path.cwd()
project_root = os.environ.get('PROJECT_ROOT') or os.environ.get('MASTERPROEF_ROOT')
if not project_root:
    if current_dir.name == 'notebooks':
        project_root = str(current_dir.parent)
    elif (current_dir / 'notebooks').exists():
        project_root = str(current_dir)
    else:
        project_root = '/Users/d1ff1cult/Local/masterproef_new'

project_root = os.path.abspath(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import ExperimentConfig, DataConfig, ModelConfig, TrainingConfig
from data import DataPipeline
from models import ProbabilisticTransformer
from core.trainer import Trainer
from core.evaluator import Evaluator
from transformations import StandardScalingTransformation

# GPU memory growth
try:
    gpus = tf.config.experimental.list_physical_devices("GPU")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
except Exception as exc:
    print(f"GPU config skipped: {exc}")

print(f"Project root: {project_root}")

2026-03-04 20:43:52.410423: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772653432.426391  377326 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772653432.431229  377326 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772653432.444107  377326 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772653432.444123  377326 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772653432.444125  377326 computation_placer.cc:177] computation placer alr

Project root: /home/d1ff1cult/masterproef_new


In [2]:
# Configuration
RESULTS_DIR = Path(project_root) / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = RESULTS_DIR / "head_study_results.json"

N_RUNS = 10
DATASET_NAME = "BE_ENTSOE"
TEST_MONTHS = 6
INPUT_WINDOW = 168
OUTPUT_HORIZON = 24
EPOCHS = 25
BATCH_SIZE = 32

configs = []

# Baselines
configs.append({"head_type": "gaussian", "head_params": {}, "name": "Gaussian"})
configs.append({"head_type": "johnson_su", "head_params": {}, "name": "JohnsonSU"})

# mixtures
for n in [1, 2, 3, 5]:
    configs.append({"head_type": "mixture_gaussian", "head_params": {"n_components": n}, "name": f"GMM-{n}"})
    configs.append({"head_type": "mixture_johnson_su", "head_params": {"n_components": n}, "name": f"JSU-Mix-{n}"})

# quantiles
for n in [9, 19, 29, 39, 49, 59, 69, 79, 89, 99]:
    q_list = np.linspace(1/(n+1), n/(n+1), n).tolist()
    configs.append({"head_type": "quantile", "head_params": {"quantiles": q_list}, "name": f"Quantile-{n}"})

print(f"Total configurations: {len(configs)}")
print(f"Total training runs: {len(configs) * N_RUNS}")

Total configurations: 20
Total training runs: 200


In [3]:
# Data loading
data_config = DataConfig(
    dataset_name=DATASET_NAME,
    test_duration_months=TEST_MONTHS,
    input_window=INPUT_WINDOW,
    output_horizon=OUTPUT_HORIZON,
)
pipeline = DataPipeline(data_config)
train_df, val_df, test_df = pipeline.get_data_splits()

X_train_raw, y_train_raw = pipeline.create_sequences(train_df)
X_val_raw, y_val_raw = pipeline.create_sequences(val_df)
X_test_raw, y_test_raw = pipeline.create_sequences(test_df)

transform = StandardScalingTransformation()
transform.fit(X_train_raw, y_train_raw)

X_train, y_train = transform.transform(X_train_raw, y_train_raw)
X_val, y_val = transform.transform(X_val_raw, y_val_raw)
X_test, y_test = transform.transform(X_test_raw, y_test_raw)

print(f"Train: {X_train.shape}, Val: {X_val.shape}")

Train: (10367, 168, 28), Val: (1997, 168, 28)


In [4]:
# Cache
def load_results():
    if CACHE_FILE.exists():
        with open(CACHE_FILE, "r") as f:
            return json.load(f)
    return {}

def save_results(results):
    with open(CACHE_FILE, "w") as f:
        json.dump(results, f, indent=2)

results_cache = load_results()

In [ ]:
# Experiment loop
for i, conf in enumerate(configs):
    conf_name = conf["name"]
    print(f"\n[{i+1}/{len(configs)}] Processing {conf_name}...")
    
    if conf_name not in results_cache:
        results_cache[conf_name] = {"runs": []}

    completed_runs = len(results_cache[conf_name]["runs"])
    if completed_runs >= N_RUNS:
        print(f"  Already completed {completed_runs} runs. Skipping.")
        continue
        
    runs_to_do = N_RUNS - completed_runs
    print(f"  Running {runs_to_do} repetitions...")
    
    for run_idx in range(runs_to_do):
        current_run_id = completed_runs + run_idx + 1
        print(f"    Run {current_run_id}/{N_RUNS}...")
        
        tf.keras.backend.clear_session()
        gc.collect()
        
        # Experiment config
        exp_config = ExperimentConfig(
            name=f"{conf_name}_run_{current_run_id}",
            data_config=data_config,
            model_config=ModelConfig(d_model=224, num_heads=4, num_layers=2), # Default config
            training_config=TrainingConfig(
                batch_size=BATCH_SIZE, 
                epochs=EPOCHS,
                learning_rate=7e-4, 
                patience=5
            ),
            head_type=conf["head_type"],
            head_params=conf["head_params"],
        )
        
        try:
            model = ProbabilisticTransformer(exp_config)
            trainer = Trainer(exp_config)
            
            start_time = time.time()
            history = trainer.train(model, X_train, y_train, X_val, y_val)
            fit_time = time.time() - start_time
            
            evaluator = Evaluator(model, transform)
            metrics = evaluator.evaluate(X_test, y_test_raw)
            
            run_result = {
                "run_id": current_run_id,
                "metrics": {k: float(v) for k, v in metrics.items()},
                "fit_time_s": fit_time
            }
            results_cache[conf_name]["runs"].append(run_result)
            save_results(results_cache)
            
            print(f"      MAE: {metrics.get('MAE', 0):.4f}")
            
        except Exception as e:
            print(f"      FAILED: {e}")
            import traceback
            traceback.print_exc()



[1/20] Processing Gaussian...
  Running 10 repetitions...
    Run 1/10...


I0000 00:00:1772653435.572389  377326 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7483 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1080, pci bus id: 0000:01:00.0, compute capability: 6.1


Epoch 1/25


I0000 00:00:1772653440.102800  377454 service.cc:152] XLA service 0x768138003e70 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772653440.102816  377454 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce GTX 1080, Compute Capability 6.1
2026-03-04 20:44:00.230632: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1772653440.935743  377454 cuda_dnn.cc:529] Loaded cuDNN version 90300


  8/324 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - loss: 1.6913

I0000 00:00:1772653447.090826  377454 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


324/324 ━━━━━━━━━━━━━━━━━━━━ 26s 49ms/step - loss: 1.1192 - val_loss: 1.2589 - learning_rate: 7.0000e-04
Epoch 2/25
324/324 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - loss: 0.7833 - val_loss: 1.2490 - learning_rate: 7.0000e-04
Epoch 3/25
 20/324 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - loss: 0.6462

In [6]:
# Analysis
# Probabilistic metrics (use .get() for backward compatibility with cached runs)
PROB_METRICS = ["MAPE", "PICP", "MPIW", "PINAW", "IntervalScore", "CRPS", "Pinball_10", "Pinball_50", "Pinball_90", "Avg_Pinball"]

summary_rows = []

for name, data in results_cache.items():
    runs = data["runs"]
    if not runs:
        continue

    maes = [r["metrics"]["MAE"] for r in runs]
    rmses = [r["metrics"]["RMSE"] for r in runs]
    times = [r["fit_time_s"] for r in runs]

    row = {
        "Configuration": name,
        "MAE_mean": np.mean(maes),
        "MAE_std": np.std(maes),
        "RMSE_mean": np.mean(rmses),
        "FitTime_mean": np.mean(times),
    }
    for m in PROB_METRICS:
        vals = [r["metrics"].get(m) for r in runs]
        vals = [v for v in vals if v is not None and not (isinstance(v, float) and np.isnan(v))]
        row[f"{m}_mean"] = np.mean(vals) if vals else np.nan
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values("MAE_mean")
display(summary_df)

,Configuration,MAE_mean,MAE_std,RMSE_mean,FitTime_mean
6,GMM-3,20.306898,0.733378,27.456014,72.009124
8,GMM-5,20.377284,0.671109,27.547588,73.321045
16,Quantile-69,20.461992,0.728226,27.521490,97.742285
3,JSU-Mix-1,20.607707,0.753274,27.706957,69.383671
1,JohnsonSU,20.638729,0.682759,27.864911,73.997707
9,JSU-Mix-5,20.656887,0.567029,28.149140,74.262315
5,JSU-Mix-2,20.751550,1.090756,27.874209,77.311268
7,JSU-Mix-3,20.765920,1.214256,28.149337,76.682427
13,Quantile-39,21.008796,0.765949,28.505527,80.007946
4,GMM-2,21.109455,1.060399,28.371530,71.288829
